# Model experiments

**Objective:** compare algorithms, feature sets and hyperparameters with the
**same validation framework** (5-fold grouped by location), then build the
submission.

Rules followed in this notebook:

- one validation setup for everything, so differences come from the models;
- no early stopping on the validation fold (it would leak information into the
  score), fixed number of trees instead;
- the threshold is chosen on out-of-fold predictions;
- every experiment gets an ID (`EXP-001`, ...) in `outputs/experiments_log.csv`;
- test predictions are the **average of the fold models**, which keeps the
  threshold and the test probabilities consistent.

## Imports

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (the notebook runs from notebooks/)
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

from src.config import ID_COL, LOCATION_COL, TARGET, TEST_MERGED, TRAIN_MERGED
from src.experiments import ExperimentLogger
from src.features import prepare_features
from src.metrics import find_best_threshold
from src.models import make_builder
from src.submission import make_submission
from src.validation import cross_validate, evaluate_oof, make_folds

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

## Load data and build folds

In [ ]:
train = pd.read_csv(TRAIN_MERGED)
test = pd.read_csv(TEST_MERGED)

y = train[TARGET]
folds = make_folds(y, "group", groups=train[LOCATION_COL])
VALIDATION_NAME = "group_location_5fold"

# Start a clean experiment log for this run
logger = ExperimentLogger()
logger.reset()

# Every fitted candidate is stored here: name -> dict(oof, test_pred, metrics, ...)
candidates = {}

## Experiment runner

In [ ]:
def run_experiment(name, feature_set="base", drop_location=True, params=None, conclusion=""):
    """Cross-validate one model, log it and keep its predictions."""
    X = prepare_features(train, feature_set, drop_location)
    X_test = prepare_features(test, feature_set, drop_location)

    result = cross_validate(make_builder(name, X, params), X, y, folds, X_test)
    metrics = evaluate_oof(y, result)

    label = feature_set + ("" if drop_location else "+location")
    exp_id = logger.log(name, label, params, VALIDATION_NAME, metrics, conclusion)

    key = f"{exp_id} {name} [{label}]"
    candidates[key] = {**result, "metrics": metrics, "model": name, "features": label}
    print(f"{exp_id} | {name:14s} | {label:18s} | AUC {metrics['ROC_AUC']:.4f} | "
          f"F1 {metrics['F1']:.4f} | score {metrics['score']:.4f}")
    return key

## A. Feature sets (CatBoost)

Does location information help, and do the derived climate features help?

In [ ]:
fs_runs = {
    "base+location": run_experiment("catboost", "base", drop_location=False,
                                    conclusion="Location variables kept"),
    "base": run_experiment("catboost", "base", drop_location=True,
                           conclusion="Location variables removed"),
    "engineered": run_experiment("catboost", "engineered", drop_location=True,
                                 conclusion="Derived climate features added"),
}

fs_scores = {k: candidates[v]["metrics"]["score"] for k, v in fs_runs.items()}
best_fs_label = max(fs_scores, key=fs_scores.get)
print("\nBest feature set:", best_fs_label)

BEST_FEATURE_SET = "engineered" if best_fs_label == "engineered" else "base"
BEST_DROP_LOCATION = best_fs_label != "base+location"

## B. Algorithm comparison

All algorithms use the best feature set from section A and default parameters.

In [ ]:
# CatBoost with default parameters was already run in section A on this feature set
model_runs = {"catboost": fs_runs[best_fs_label]}

for model_name in ["logreg", "random_forest", "extra_trees", "lightgbm", "xgboost"]:
    model_runs[model_name] = run_experiment(
        model_name, BEST_FEATURE_SET, BEST_DROP_LOCATION, conclusion="Default parameters"
    )

## C. Hyperparameters for every model

Each algorithm gets the same tuning effort: 4 configurations on top of its default run
(section A/B), all with the same folds and a fixed number of trees (no early stopping).
The default run of each model is already in the log, so any gain is measured against it.

In [ ]:
TUNING_GRIDS = {
    "catboost": [
        {"depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 5},
        {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
        {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 8},
        {"depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 10, "iterations": 900},
    ],
    "logreg": [
        {"C": 0.03},
        {"C": 0.1},
        {"C": 3.0},
        {"C": 10.0},
    ],
    "random_forest": [
        {"n_estimators": 500, "max_depth": 8, "min_samples_leaf": 5},
        {"n_estimators": 500, "max_depth": 12, "min_samples_leaf": 3},
        {"n_estimators": 500, "min_samples_leaf": 10, "max_features": 0.5},
        {"n_estimators": 800, "max_depth": 6, "min_samples_leaf": 8, "max_features": "sqrt"},
    ],
    "extra_trees": [
        {"n_estimators": 500, "max_depth": 8, "min_samples_leaf": 5},
        {"n_estimators": 500, "max_depth": 12, "min_samples_leaf": 3},
        {"n_estimators": 500, "min_samples_leaf": 10, "max_features": 0.5},
        {"n_estimators": 800, "max_depth": 6, "min_samples_leaf": 8, "max_features": "sqrt"},
    ],
    "lightgbm": [
        {"n_estimators": 400, "learning_rate": 0.03, "num_leaves": 15, "min_child_samples": 30,
         "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8, "reg_lambda": 5},
        {"n_estimators": 600, "learning_rate": 0.02, "num_leaves": 7, "min_child_samples": 40,
         "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.7, "reg_lambda": 10},
        {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 20,
         "subsample": 0.9, "subsample_freq": 1, "colsample_bytree": 0.9, "reg_lambda": 1},
        {"n_estimators": 800, "learning_rate": 0.01, "num_leaves": 10, "min_child_samples": 50,
         "subsample": 0.7, "subsample_freq": 1, "colsample_bytree": 0.6, "reg_lambda": 15},
    ],
    "xgboost": [
        {"n_estimators": 400, "learning_rate": 0.03, "max_depth": 3, "min_child_weight": 5,
         "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 5},
        {"n_estimators": 600, "learning_rate": 0.02, "max_depth": 4, "min_child_weight": 10,
         "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 10},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5, "min_child_weight": 3,
         "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1},
        {"n_estimators": 800, "learning_rate": 0.01, "max_depth": 3, "min_child_weight": 15,
         "subsample": 0.7, "colsample_bytree": 0.6, "reg_lambda": 15},
    ],
}

tuned_runs = {}
for model_name, configs in TUNING_GRIDS.items():
    tuned_runs[model_name] = [
        run_experiment(model_name, BEST_FEATURE_SET, BEST_DROP_LOCATION, params=cfg,
                       conclusion="Hyperparameter search")
        for cfg in configs
    ]

## Experiment leaderboard

In [ ]:
log = logger.load()

# Comparison columns, computed against the default run of the same model
# on the selected feature set
label = BEST_FEATURE_SET + ("" if BEST_DROP_LOCATION else "+location")
is_default = log["hyperparameters"].astype(str).isin(["{}", "", "nan", "None"])
defaults = (log[is_default & (log["feature_set"] == label)]
            .set_index("model")["score"])

log["default_score"] = log["model"].map(defaults)
log["gain_vs_default"] = (log["score"] - log["default_score"]).round(4)
log["gap_to_best"] = (log["score"] - log["score"].max()).round(4)

cols = ["exp_id", "model", "feature_set", "hyperparameters", "threshold",
        "F1", "ROC_AUC", "score", "fold_auc_std",
        "default_score", "gain_vs_default", "gap_to_best"]
leaderboard = log[cols].sort_values("score", ascending=False).reset_index(drop=True)
leaderboard

**How to read the leaderboard:** compare score gaps with `fold_auc_std`.
When two models differ by less than the fold-to-fold spread, treat them as a
tie rather than a real improvement, and prefer the simpler or more stable one.

**Comparison columns:** `gain_vs_default` is the score change of a configuration versus the
default run of the same algorithm; `gap_to_best` is the distance to the best score of the
whole log (0 = best). Per-model summary below.

In [ ]:
summary = (
    log[log["feature_set"].isin([label, "mixed"]) | log["model"].str.startswith("ensemble")]
    .groupby("model")
    .agg(default=("default_score", "first"),
         best=("score", "max"),
         n_runs=("score", "size"))
)
summary["gain"] = (summary["best"] - summary["default"]).round(4)
summary.sort_values("best", ascending=False)

## D. Feature importance (best single model)

Diagnostic only: importance is not causality, but a dominant or odd variable
is a hint to look for leakage.

In [ ]:
best_single_key = max(
    (k for k in candidates if candidates[k]["model"] != "logreg"),
    key=lambda k: candidates[k]["metrics"]["score"],
)
print("Best single model:", best_single_key)

X_full = prepare_features(train, BEST_FEATURE_SET, BEST_DROP_LOCATION)
best_model_name = candidates[best_single_key]["model"]
importance_model = make_builder(best_model_name, X_full)()
importance_model.fit(X_full, y)

if best_model_name == "catboost":
    importances = pd.Series(importance_model.get_feature_importance(), index=X_full.columns)
elif hasattr(importance_model[-1], "feature_importances_"):
    names = importance_model[0].get_feature_names_out()
    importances = pd.Series(importance_model[-1].feature_importances_, index=names)
else:
    importances = pd.Series(dtype=float)

if len(importances):
    importances.sort_values().tail(15).plot(kind="barh", figsize=(7, 5))
    plt.title("Top 15 feature importances")
    plt.show()

## E. Probability comparison and ensembles

Models with similar scores can still make different errors. The correlation
of their out-of-fold probabilities shows how much an ensemble can gain.

In [ ]:
# Best configuration per algorithm
best_per_model = {}
for key, c in candidates.items():
    name = c["model"]
    if name not in best_per_model or c["metrics"]["score"] > candidates[best_per_model[name]]["metrics"]["score"]:
        best_per_model[name] = key

top_keys = sorted(best_per_model.values(), key=lambda k: -candidates[k]["metrics"]["score"])[:3]
print("Models used in the ensemble:")
for k in top_keys:
    print("  ", k)

oof_matrix = pd.DataFrame({k.split(" ")[1]: candidates[k]["oof"] for k in top_keys})
display(oof_matrix.corr().round(3))

In [ ]:
def blend(keys, method):
    """Blend several candidates by mean probability or by mean rank."""
    oofs = np.column_stack([candidates[k]["oof"] for k in keys])
    tests = np.column_stack([candidates[k]["test_pred"] for k in keys])

    if method == "mean":
        oof, test_pred = oofs.mean(axis=1), tests.mean(axis=1)
    else:  # rank average, rescaled to [0, 1]
        oof = np.column_stack([rankdata(oofs[:, i]) / len(oofs) for i in range(oofs.shape[1])]).mean(axis=1)
        test_pred = np.column_stack([rankdata(tests[:, i]) / len(tests) for i in range(tests.shape[1])]).mean(axis=1)

    fold_auc = [roc_auc_score(y.iloc[v], oof[v]) for _, v in folds]
    result = {"oof": oof, "test_pred": test_pred, "fold_auc": fold_auc,
              "covered": np.ones(len(oof), dtype=bool)}
    return result, evaluate_oof(y, result)


for method in ["mean", "rank"]:
    result, metrics = blend(top_keys, method)
    exp_id = logger.log(
        f"ensemble_{method}", "mixed", {"members": [k.split(" ")[1] for k in top_keys]},
        VALIDATION_NAME, metrics, "Simple blend of the top 3 models",
    )
    key = f"{exp_id} ensemble_{method}"
    candidates[key] = {**result, "metrics": metrics, "model": f"ensemble_{method}", "features": "mixed"}
    print(f"{exp_id} | ensemble_{method:5s} | AUC {metrics['ROC_AUC']:.4f} | "
          f"F1 {metrics['F1']:.4f} | score {metrics['score']:.4f}")

## F. Final choice and submission

The final model is the **best-scoring single model** across all experiments (all tuned
configurations included). Ensembles are excluded by default (`INCLUDE_ENSEMBLES = False`):
when an ensemble beats the best single model by less than the fold-to-fold spread, the
simpler model is preferred. Set the flag to `True` to let the mean blend compete.
Rank blends stay excluded because their test scores are rescaled by rank, which does not
match the threshold learned on out-of-fold ranks.

Note that picking the maximum over many candidates is slightly optimistic.

In [ ]:
INCLUDE_ENSEMBLES = False

eligible = {
    k: c for k, c in candidates.items()
    if "ensemble_rank" not in k
    and (INCLUDE_ENSEMBLES or not c["model"].startswith("ensemble"))
}
ranking = sorted(eligible, key=lambda k: eligible[k]["metrics"]["score"], reverse=True)

print("Top 5 eligible candidates:")
for k in ranking[:5]:
    m = eligible[k]["metrics"]
    print(f"  {k:55s} score {m['score']:.4f} | fold_auc_std {m['fold_auc_std']:.4f}")

final_key = ranking[0]
final = candidates[final_key]

print("\nFinal candidate:", final_key)
print({k: round(v, 4) for k, v in final["metrics"].items() if k not in ("table", "n_scored")})

mask = final["covered"]
best_threshold, threshold_table = find_best_threshold(y[mask], final["oof"][mask])
print("\nThreshold used:", best_threshold["threshold"])

In [ ]:
submission, path = make_submission(
    ids=test[ID_COL],
    test_prob=final["test_pred"],
    threshold=best_threshold["threshold"],
    filename="submission_v2.csv",
)

print("Saved to:", path)
print("Shape:", submission.shape)
print("\nPredicted label distribution (test):")
print(submission["TargetF1"].value_counts(normalize=True).round(3))
print("\nTrain positive rate:", round(y.mean(), 3))
print("\nProbability summary:")
print(submission["TargetRAUC"].describe().round(3))
submission.head()